# Chapter 7: Blockwise Attention Path

This notebook shows the IO-aware rewrite used as the first structural optimization in Chapter 7.

## What this notebook teaches

- How blockwise online softmax changes the cost structure
- How to compare blockwise attention against the reference path
- How to keep the benchmark record and the chapter narrative aligned

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "HANDOFF.md").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np

from examples.benchmark_record_template import set_benchmark_record
from examples.ch07_attention_core import attention_blockwise, attention_reference, benchmark, make_attention_inputs

np.random.seed(0)
batch, seq_len, dim = 1, 32, 64
block_size = 8
q, k, v = make_attention_inputs(batch, seq_len, dim)
ref, ref_ms = benchmark(attention_reference, q, k, v, runs=20)
out, elapsed_ms = benchmark(attention_blockwise, q, k, v, runs=20, block_size=block_size)

print('Chapter 7: Blockwise Attention')
print('=' * 72)
print(f'Shape: B={batch}, S={seq_len}, D={dim}')
print(f'Block size: {block_size}')
print(f'Reference latency: {ref_ms:.4f} ms per run')
print(f'Blockwise latency:  {elapsed_ms:.4f} ms per run')
print(f'Max error: {np.max(np.abs(ref - out)):.2e}')
print()
print('Interpretation:')
print('- This version keeps only running statistics on the fly.')
print('- It demonstrates the IO-reduction idea behind FlashAttention.')
print()
record = set_benchmark_record(
    scenario='chapter 7 attention blockwise',
    operator='Attention',
    platform='CPU',
    target='numpy',
    dtype='float32',
    shape=f'B={batch},S={seq_len},D={dim}',
    baseline='standard attention',
    optimization=f'blockwise online softmax (block_size={block_size})',
)
print('Benchmark record skeleton:')
for key in ['scenario', 'operator', 'platform', 'target', 'dtype', 'shape', 'baseline', 'optimization']:
    print(f'  {key}: {record[key]}')


## Takeaway

This notebook makes the IO-reduction step explicit before the chapter moves on to fused attention and longer scheduling chains.